In [ ]:
import os, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from scipy import signal
from IPython.display import Audio, display

In [ ]:


warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100


AUDIO_PATH = r"C:\academics\codes\DSIP\Yeshanagula The Paradise Nani Keerthy Suresh Anirudh Ravichander Srikanth Odela.mp3"
OUT_DIR    = "output_audio"         
MAX_SECONDS = None                   
os.makedirs(OUT_DIR, exist_ok=True)



In [ ]:

y, sr = librosa.load(AUDIO_PATH, sr=None, mono=True, duration=MAX_SECONDS)
nyq = sr / 2
print(f"File          : {os.path.basename(AUDIO_PATH)}")
print(f"Sample rate   : {sr} Hz   (Nyquist = {nyq:.0f} Hz)")
print(f"Duration      : {len(y)/sr:.2f} s")
display(Audio(y, rate=sr))

In [ ]:
ORDER = 6  

def design_lowpass(cutoff, fs, order=ORDER):
    return signal.butter(order, cutoff, btype="low", fs=fs, output="sos")

def design_bandpass(low, high, fs, order=ORDER):
    return signal.butter(order, [low, high], btype="band", fs=fs, output="sos")

def design_bandstop(low, high, fs, order=ORDER):
    return signal.butter(order, [low, high], btype="bandstop", fs=fs, output="sos")

def apply_filter(sos, x):
    return signal.sosfiltfilt(sos, x)          # zero-phase filtering

def save_mp3(x, fs, path, normalize=True):
    """Peak-normalise (to avoid clipping) and save as MP3.
    Tries libsndfile's MP3 writer first, falls back to ffmpeg."""
    x = np.asarray(x, dtype=np.float32)
    if normalize and np.max(np.abs(x)) > 0:
        x = 0.95 * x / np.max(np.abs(x))
    try:
        sf.write(path, x, fs, format="MP3")
    except Exception:
        wav = path.replace(".mp3", "_tmp.wav")
        sf.write(wav, x, fs)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav,
                        "-codec:a", "libmp3lame", "-q:a", "2", path], check=True)
        os.remove(wav)
    print("saved ->", path)

def metrics(x, ref, fs):
    """Numerical description of a filtered signal compared with the original."""
    S = np.abs(librosa.stft(x)) ** 2
    return {
        "RMS level (dB re original)": 10*np.log10(np.sum(x**2) / np.sum(ref**2) + 1e-12),
        "Energy retained (%)":        100 * np.sum(x**2) / np.sum(ref**2),
        "Spectral centroid (Hz)":     float(np.mean(librosa.feature.spectral_centroid(S=np.sqrt(S), sr=fs))),
        "Spectral rolloff 95% (Hz)":  float(np.mean(librosa.feature.spectral_rolloff(S=np.sqrt(S), sr=fs, roll_percent=0.95))),
    }

REF_MAG = np.abs(librosa.stft(y)).max()   # common reference so all spectrograms share one dB scale

def plot_spectrogram(x, fs, title, fmax=None):
    D = librosa.amplitude_to_db(np.abs(librosa.stft(x)), ref=REF_MAG, top_db=None)
    plt.figure(figsize=(10, 3.6))
    librosa.display.specshow(D, sr=fs, x_axis="time", y_axis="hz", cmap="magma", vmin=-80, vmax=0)
    plt.colorbar(format="%+2.0f dB")
    if fmax: plt.ylim(0, fmax)
    plt.title(title); plt.xlabel("Time (s)"); plt.ylabel("Frequency (Hz)")
    plt.tight_layout(); plt.show()

def plot_filter_responses(sos_dict, fs, title):
    plt.figure(figsize=(9, 4))
    for label, sos in sos_dict.items():
        w, h = signal.sosfreqz(sos, worN=8192, fs=fs)
        plt.semilogx(w, 20*np.log10(np.abs(h) + 1e-12), label=label)
    plt.axhline(-3, color="gray", ls="--", lw=0.8, label="-3 dB")
    plt.ylim(-80, 5); plt.xlim(20, fs/2)
    plt.grid(True, which="both", alpha=0.3)
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Magnitude (dB)")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

def plot_psd(sig_dict, fs, title):
    plt.figure(figsize=(9, 4))
    for label, x in sig_dict.items():
        f, p = signal.welch(x, fs, nperseg=4096)
        plt.semilogx(f, 10*np.log10(p + 1e-14), label=label, lw=1)
    plt.xlim(20, fs/2); plt.grid(True, which="both", alpha=0.3)
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Power spectral density (dB/Hz)")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plot_spectrogram(y, sr, "Original audio: full spectrum")
plot_psd({"Original": y}, sr, "Original audio: average power spectrum")

In [ ]:
CUTOFFS = [500, 1500, 3000, 6000]   # Hz  (at least three required; four used)

lp_sos = {f"LPF {fc} Hz": design_lowpass(fc, sr) for fc in CUTOFFS}
plot_filter_responses(lp_sos, sr, f"Low-pass filters: Butterworth order {ORDER} (single pass)")

In [ ]:
lp_out = {}
rows = [{"Signal": "Original", **metrics(y, y, sr)}]

for fc, (label, sos) in zip(CUTOFFS, lp_sos.items()):
    yf = apply_filter(sos, y)
    lp_out[fc] = yf
    save_mp3(yf, sr, f"{OUT_DIR}/lowpass_{fc}Hz.mp3")
    rows.append({"Signal": label, **metrics(yf, y, sr)})

lp_table = pd.DataFrame(rows).set_index("Signal").round(2)
lp_table

In [ ]:
# Spectrograms of every low-pass result (same colour scale as the original)
for fc, yf in lp_out.items():
    plot_spectrogram(yf, sr, f"Low-pass filtered, cutoff = {fc} Hz")

In [ ]:
plot_psd({"Original": y, **{f"LPF {fc} Hz": v for fc, v in lp_out.items()}}, sr,
         "Average power spectrum: original vs low-pass filtered")

In [ ]:
# Listen and compare
print("Original");  display(Audio(y, rate=sr))
for fc, yf in lp_out.items():
    print(f"Low-pass @ {fc} Hz"); display(Audio(yf, rate=sr))

In [ ]:
VOCAL_LOW, VOCAL_HIGH = 300, 3400   # Hz  (adjust, e.g. 200-5000, to experiment)

bp_sos = design_bandpass(VOCAL_LOW, VOCAL_HIGH, sr)
bs_sos = design_bandstop(VOCAL_LOW, VOCAL_HIGH, sr)

plot_filter_responses({f"Band-pass {VOCAL_LOW}-{VOCAL_HIGH} Hz  (vocals kept)": bp_sos,
                       f"Band-stop {VOCAL_LOW}-{VOCAL_HIGH} Hz  (vocals suppressed)": bs_sos},
                      sr, "Filters used for vocal / background separation")

In [ ]:
vocals_kept       = apply_filter(bp_sos, y)   # background suppressed
background_kept   = apply_filter(bs_sos, y)   # vocals suppressed

save_mp3(vocals_kept,     sr, f"{OUT_DIR}/bandpass_vocals_background_suppressed.mp3")
save_mp3(background_kept, sr, f"{OUT_DIR}/bandstop_background_vocals_suppressed.mp3")

rows = [
    {"Signal": "Original", **metrics(y, y, sr)},
    {"Signal": f"Band-pass {VOCAL_LOW}-{VOCAL_HIGH} Hz (vocal band)", **metrics(vocals_kept, y, sr)},
    {"Signal": f"Band-stop {VOCAL_LOW}-{VOCAL_HIGH} Hz (background)",  **metrics(background_kept, y, sr)},
]
bp_table = pd.DataFrame(rows).set_index("Signal").round(2)
bp_table

In [ ]:
plot_spectrogram(y, sr, "Original audio")
plot_spectrogram(vocals_kept, sr, f"Band-pass {VOCAL_LOW}-{VOCAL_HIGH} Hz: vocals emphasised, background suppressed")
plot_spectrogram(background_kept, sr, f"Band-stop {VOCAL_LOW}-{VOCAL_HIGH} Hz: background emphasised, vocals suppressed")

In [ ]:
plot_psd({"Original": y, "Vocal band (band-pass)": vocals_kept, "Background (band-stop)": background_kept},
         sr, "Average power spectrum: original vs separated components")

# Waveform comparison of a short excerpt
t0, t1 = 0, min(5, len(y)/sr)
t = np.arange(len(y)) / sr
m = (t >= t0) & (t < t1)
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for a, (lab, x) in zip(ax, [("Original", y), ("Vocal band", vocals_kept), ("Background", background_kept)]):
    a.plot(t[m], x[m], lw=0.5); a.set_ylabel(lab)
ax[-1].set_xlabel("Time (s)"); fig.suptitle("Waveforms of the first seconds"); plt.tight_layout(); plt.show()

In [ ]:
print("Original");                         display(Audio(y, rate=sr))
print("Band-pass: vocals kept");            display(Audio(vocals_kept, rate=sr))
print("Band-stop: vocals suppressed");      display(Audio(background_kept, rate=sr))

In [ ]:
variants = {"200-5000 Hz": (200, 5000), "300-3400 Hz": (300, 3400), "500-2500 Hz": (500, 2500)}
rows = []
for name, (lo, hi) in variants.items():
    v = apply_filter(design_bandpass(lo, hi, sr), y)
    b = apply_filter(design_bandstop(lo, hi, sr), y)
    rows.append({"Vocal band": name,
                 "Band-pass energy kept (%)": 100*np.sum(v**2)/np.sum(y**2),
                 "Band-stop energy kept (%)": 100*np.sum(b**2)/np.sum(y**2)})
pd.DataFrame(rows).set_index("Vocal band").round(1)

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:55s} {os.path.getsize(os.path.join(OUT_DIR, f))/1024:8.1f} KB")


